# Time-Aware Sample Weighting

This experiment tests whether exponential recency weighting improves the current 463-feature LightGBM pipeline. Half-life selection uses three expanding folds inside TRAIN. External VALIDATION is evaluated once after selection, and HOLDOUT is never assigned or accessed.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
    validate_training_data,
)
from fraud_detection.models import (
    evaluate_probabilities,
    exponential_time_decay_weights,
    make_expanding_time_folds,
    prepare_engineered_matrices,
)

notebook_started = time.perf_counter()

## Weighting formula

For each fit portion only, `age_days = (max_fit_TransactionDT - TransactionDT) / 86400` and `raw_weight = exp(-ln(2) * age_days / half_life_days)`. Raw weights are divided by their fit-set mean. No floor is needed for the observed time ranges, and evaluation timestamps never influence the reference time.

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validate_training_data(train)
del transaction, identity, merged
gc.collect()

folds = make_expanding_time_folds(train, n_splits=3, initial_train_fraction=0.70)
fold_design = pd.DataFrame(
    [
        {
            "Fold": number,
            "Fit Rows": len(fold.train_indices),
            "Evaluation Rows": len(fold.evaluation_indices),
            "Fit DT Max": train.iloc[fold.train_indices]["TransactionDT"].max(),
            "Evaluation DT Min": train.iloc[fold.evaluation_indices]["TransactionDT"].min(),
            "Evaluation DT Max": train.iloc[fold.evaluation_indices]["TransactionDT"].max(),
        }
        for number, fold in enumerate(folds, start=1)
    ]
)
weighting_candidates = {
    "unweighted": None,
    "half_life_30_days": 30.0,
    "half_life_60_days": 60.0,
    "half_life_90_days": 90.0,
}
lightgbm_parameters = {
    "learning_rate": 0.1,
    "n_estimators": 400,
    "num_leaves": 63,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}
display(fold_design)
print("Feature pipeline: frequency + five historical amounts + 20 composite features")
print("External validation status: not assigned or accessed before CV selection")
print("Final holdout status: not assigned or accessed")

,Fold,Fit Rows,Evaluation Rows,Fit DT Max,Evaluation DT Min,Evaluation DT Max
0,1,289364,41338,7135483,7135609,8121450
1,2,330702,41338,8121450,8121466,9247568
2,3,372040,41338,9247568,9247583,10437996


Feature pipeline: frequency + five historical amounts + 20 composite features
External validation status: not assigned or accessed before CV selection
Final holdout status: not assigned or accessed


## Leakage-safe TRAIN-only temporal CV

Every fold independently fits frequency maps, historical state, imputation, and scaling. Sample weights use only that fold's fit timestamps. The same prepared matrix and fixed LightGBM configuration are used for all four candidates.

In [3]:
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-time-weighting")
cv_metrics = {name: [] for name in weighting_candidates}
weight_statistics = {name: [] for name in weighting_candidates}
fit_seconds = {name: [] for name in weighting_candidates}
preparation_seconds = []
prepared_memory_mib = []
feature_count = None

for fold_number, fold in enumerate(folds, start=1):
    fit_frame = train.iloc[fold.train_indices]
    evaluation_frame = train.iloc[fold.evaluation_indices]
    preparation_started = time.perf_counter()
    prepared = prepare_engineered_matrices(
        fit_frame,
        evaluation_frame,
        include_graph=False,
        include_composite=True,
        scaling_batch_rows=len(fit_frame),
    )
    preparation_seconds.append(time.perf_counter() - preparation_started)
    prepared_memory_mib.append((prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2)
    feature_count = prepared.feature_count
    y_fit = fit_frame["isFraud"].astype("int8")
    y_evaluation = evaluation_frame["isFraud"].astype("int8")

    for strategy, half_life_days in weighting_candidates.items():
        if half_life_days is None:
            weights = np.ones(len(fit_frame), dtype=np.float64)
            model_weights = None
        else:
            weights = exponential_time_decay_weights(
                fit_frame["TransactionDT"], half_life_days=half_life_days
            )
            model_weights = weights
        weight_statistics[strategy].append(
            {
                "minimum": float(np.min(weights)),
                "median": float(np.median(weights)),
                "maximum": float(np.max(weights)),
                "mean": float(np.mean(weights)),
            }
        )
        fit_started = time.perf_counter()
        model = lgb.LGBMClassifier(**lightgbm_parameters)
        model.fit(prepared.x_train, y_fit, sample_weight=model_weights)
        probabilities = model.predict_proba(prepared.x_evaluation)[:, 1]
        fit_seconds[strategy].append(time.perf_counter() - fit_started)
        metrics = evaluate_probabilities(y_evaluation, probabilities)
        cv_metrics[strategy].append(metrics)
        print(
            f"fold={fold_number} strategy={strategy} PR-AUC={metrics.pr_auc:.6f} "
            f"weight_min={weights.min():.4f} weight_max={weights.max():.4f}"
        )
        del model, probabilities, weights, model_weights
    del prepared
    gc.collect()

cv_rows = []
cv_run_ids = {}
for strategy, half_life_days in weighting_candidates.items():
    fold_pr_auc = [metric.pr_auc for metric in cv_metrics[strategy]]
    fold_roc_auc = [metric.roc_auc for metric in cv_metrics[strategy]]
    mean_pr_auc = float(np.mean(fold_pr_auc))
    std_pr_auc = float(np.std(fold_pr_auc, ddof=0))
    mean_roc_auc = float(np.mean(fold_roc_auc))
    with mlflow.start_run(run_name=f"time-weighting-cv-{strategy}") as run:
        mlflow.set_tags(
            {
                "stage": "time-weighting",
                "dataset": "ieee-cis",
                "selection": "temporal-cv",
            }
        )
        mlflow.log_params(
            {
                "weighting_strategy": strategy,
                "half_life_days": "none" if half_life_days is None else half_life_days,
                "feature_count": feature_count,
                **lightgbm_parameters,
            }
        )
        mlflow.log_metrics(
            {
                "fold_1_pr_auc": fold_pr_auc[0],
                "fold_2_pr_auc": fold_pr_auc[1],
                "fold_3_pr_auc": fold_pr_auc[2],
                "mean_cv_pr_auc": mean_pr_auc,
                "std_cv_pr_auc": std_pr_auc,
                "mean_cv_roc_auc": mean_roc_auc,
                "mean_fit_seconds": float(np.mean(fit_seconds[strategy])),
            }
        )
        cv_run_ids[strategy] = run.info.run_id
    cv_rows.append(
        {
            "Strategy": strategy,
            "Half-Life Days": half_life_days,
            "Fold PR-AUC": tuple(fold_pr_auc),
            "Mean CV PR-AUC": mean_pr_auc,
            "Std CV PR-AUC": std_pr_auc,
            "Mean CV ROC-AUC": mean_roc_auc,
            "Mean Fit Seconds": float(np.mean(fit_seconds[strategy])),
            "MLflow Run ID": cv_run_ids[strategy],
        }
    )
cv_results = pd.DataFrame(cv_rows)
baseline_mean = float(
    cv_results.loc[cv_results["Strategy"] == "unweighted", "Mean CV PR-AUC"].iloc[0]
)
best_weighted = (
    cv_results.loc[cv_results["Strategy"] != "unweighted"]
    .sort_values("Mean CV PR-AUC", ascending=False)
    .iloc[0]
)
if float(best_weighted["Mean CV PR-AUC"]) > baseline_mean:
    winner = best_weighted
else:
    winner = cv_results.loc[cv_results["Strategy"] == "unweighted"].iloc[0]
cv_results = cv_results.sort_values("Mean CV PR-AUC", ascending=False).reset_index(drop=True)
display(cv_results)
print(f"TRAIN-CV winner: {winner['Strategy']}")

2026/10/07 09:09:07 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-time-weighting' does not exist. Creating a new experiment.


fold=1 strategy=unweighted PR-AUC=0.554229 weight_min=1.0000 weight_max=1.0000


fold=1 strategy=half_life_30_days PR-AUC=0.556336 weight_min=0.3668 weight_max=2.4162


fold=1 strategy=half_life_60_days PR-AUC=0.554720 weight_min=0.6285 weight_max=1.6130


fold=1 strategy=half_life_90_days PR-AUC=0.575379 weight_min=0.7398 weight_max=1.3869


...

fold=3 strategy=half_life_30_days PR-AUC=0.612671 weight_min=0.2450 weight_max=2.8389


fold=3 strategy=half_life_60_days PR-AUC=0.622962 weight_min=0.5275 weight_max=1.7957


fold=3 strategy=half_life_90_days PR-AUC=0.615404 weight_min=0.6626 weight_max=1.4994


,Strategy,Half-Life Days,Fold PR-AUC,Mean CV PR-AUC,Std CV PR-AUC,Mean CV ROC-AUC,Mean Fit Seconds,MLflow Run ID
0,unweighted,NaN,"(0.5542290843797867, 0.5748513290293689, 0.625...",0.584875,0.029965,0.906202,29.809155,789682d8e85c4fd295887d713c3a1556
1,half_life_90_days,90.0,"(0.5753792091437989, 0.5569121986044531, 0.615...",0.582565,0.024414,0.903198,31.051278,087bcba5344a45fcb4295931a68a51ea
2,half_life_60_days,60.0,"(0.5547203174079476, 0.5686846167265776, 0.622...",0.582122,0.029435,0.903318,29.897532,1618fcb388f64b3bb1e4e4cc1c19c6b3
3,half_life_30_days,30.0,"(0.5563361595570846, 0.5567319917943031, 0.612...",0.575246,0.026464,0.900510,28.347494,ca01d155d6a5495ca0d93c3b58f09aba


TRAIN-CV winner: unweighted


## One external VALIDATION evaluation

Only the TRAIN-CV winner is applied below. Full-TRAIN timestamps define any final decay weights; the VALIDATION period does not contribute to their reference time.

In [4]:
validation = split.validation
final_started = time.perf_counter()
prepared_final = prepare_engineered_matrices(
    train,
    validation,
    include_graph=False,
    include_composite=True,
    scaling_batch_rows=len(train),
)
winning_strategy = str(winner["Strategy"])
winning_half_life = weighting_candidates[winning_strategy]
if winning_half_life is None:
    final_weights = None
    final_weight_values = np.ones(len(train), dtype=np.float64)
else:
    final_weight_values = exponential_time_decay_weights(
        train["TransactionDT"], half_life_days=winning_half_life
    )
    final_weights = final_weight_values
final_model = lgb.LGBMClassifier(**lightgbm_parameters)
final_model.fit(
    prepared_final.x_train,
    train["isFraud"].astype("int8"),
    sample_weight=final_weights,
)
validation_probabilities = final_model.predict_proba(prepared_final.x_evaluation)[:, 1]
validation_metrics = evaluate_probabilities(
    validation["isFraud"].astype("int8"), validation_probabilities
)
final_seconds = time.perf_counter() - final_started
final_memory_mib = (
    prepared_final.x_train.nbytes + prepared_final.x_evaluation.nbytes + final_weight_values.nbytes
) / 1024**2
final_weight_stats = {
    "minimum": float(np.min(final_weight_values)),
    "median": float(np.median(final_weight_values)),
    "maximum": float(np.max(final_weight_values)),
    "mean": float(np.mean(final_weight_values)),
}
with mlflow.start_run(run_name="time-weighting-final-winner") as run:
    mlflow.set_tags(
        {
            "stage": "time-weighting",
            "dataset": "ieee-cis",
            "selection": "temporal-cv",
            "final_refit": "true",
        }
    )
    mlflow.log_params(
        {
            "weighting_strategy": winning_strategy,
            "half_life_days": ("none" if winning_half_life is None else winning_half_life),
            "feature_count": prepared_final.feature_count,
            **lightgbm_parameters,
        }
    )
    mlflow.log_metrics(
        {
            "validation_pr_auc": validation_metrics.pr_auc,
            "validation_roc_auc": validation_metrics.roc_auc,
            "validation_precision_at_0_50": validation_metrics.precision,
            "validation_recall_at_0_50": validation_metrics.recall,
            "validation_f1_at_0_50": validation_metrics.f1,
            "weight_minimum": final_weight_stats["minimum"],
            "weight_median": final_weight_stats["median"],
            "weight_maximum": final_weight_stats["maximum"],
            "weight_mean": final_weight_stats["mean"],
        }
    )
    final_run_id = run.info.run_id

## Results and conclusion

In [5]:
previous_pr_auc = 0.595387
absolute_change = validation_metrics.pr_auc - previous_pr_auc
relative_change = absolute_change / previous_pr_auc
total_seconds = time.perf_counter() - notebook_started
mean_fit_seconds = {name: round(float(np.mean(values)), 1) for name, values in fit_seconds.items()}


def markdown_cv_table(frame):
    rows = [
        "| Strategy | Half-life | Fold PR-AUCs | Mean | Std | Mean ROC-AUC | Mean fit s |",
        "| --- | ---: | --- | ---: | ---: | ---: | ---: |",
    ]
    for _, row in frame.iterrows():
        folds_text = " / ".join(f"{value:.6f}" for value in row["Fold PR-AUC"])
        half_life = "none" if pd.isna(row["Half-Life Days"]) else f"{row['Half-Life Days']:.0f}"
        rows.append(
            f"| {row['Strategy']} | {half_life} | {folds_text} | "
            f"{row['Mean CV PR-AUC']:.6f} | {row['Std CV PR-AUC']:.6f} | "
            f"{row['Mean CV ROC-AUC']:.6f} | {row['Mean Fit Seconds']:.1f} |"
        )
    return "\n".join(rows)


winner_weight_rows = [
    "| Fold | Minimum | Median | Maximum | Mean |",
    "| ---: | ---: | ---: | ---: | ---: |",
]
for number, stats in enumerate(weight_statistics[winning_strategy], start=1):
    winner_weight_rows.append(
        f"| {number} | {stats['minimum']:.6f} | {stats['median']:.6f} | "
        f"{stats['maximum']:.6f} | {stats['mean']:.6f} |"
    )
winner_weight_table = "\n".join(winner_weight_rows)
run_lines = "\n".join(f"- **{name}:** `{run_id}`" for name, run_id in cv_run_ids.items())
change_word = "improvement" if absolute_change > 0 else "degradation"
half_life_text = "none" if winning_half_life is None else f"{winning_half_life:.0f} days"
report = f"""# Time-Aware Sample Weighting

## Formula and normalization

For each fit portion, `age_days = (max_fit_TransactionDT - TransactionDT) / 86400` and
`raw_weight = exp(-ln(2) * age_days / half_life_days)`. Weights are divided by the raw
fit-set mean, producing mean one while preserving relative ratios. No lower floor is applied.
Only fit timestamps determine the reference time.

## TRAIN-only temporal CV

Three expanding chronological folds rebuild the 463-feature pipeline independently. The pipeline
contains existing frequency encodings, five historical amount features, and all 20 composite
identity features. The fixed LightGBM configuration remains unchanged across candidates.

{markdown_cv_table(cv_results)}

- Selected strategy: **{winning_strategy}**
- Selected half-life: **{half_life_text}**

## Winner fold weight statistics

{winner_weight_table}

Full-TRAIN weights: minimum **{final_weight_stats["minimum"]:.6f}**, median
**{final_weight_stats["median"]:.6f}**, maximum **{final_weight_stats["maximum"]:.6f}**,
mean **{final_weight_stats["mean"]:.6f}**.

## External validation

- PR-AUC: **{validation_metrics.pr_auc:.6f}**
- ROC-AUC: **{validation_metrics.roc_auc:.6f}**
- Precision at 0.50: **{validation_metrics.precision:.6f}**
- Recall at 0.50: **{validation_metrics.recall:.6f}**
- F1 at 0.50: **{validation_metrics.f1:.6f}**
- Confusion matrix: **{validation_metrics.confusion_matrix}**
- Previous preferred PR-AUC: **{previous_pr_auc:.6f}**
- Absolute {change_word}: **{absolute_change:+.6f}**
- Relative {change_word}: **{relative_change:+.2%}**

## Runtime and memory

- Fold feature-preparation seconds: **{", ".join(f"{value:.1f}" for value in preparation_seconds)}**
- Mean fit seconds by strategy: **{json.dumps(mean_fit_seconds)}**
- Final TRAIN/VALIDATION workflow: **{final_seconds:.1f} seconds**
- Peak prepared matrix estimate: **{max([*prepared_memory_mib, final_memory_mib]):.1f} MiB**
- Total notebook time before report write: **{total_seconds:.1f} seconds**

## MLflow

- Experiment: `fraud-detection-time-weighting`
{run_lines}
- **Final winner:** `{final_run_id}`

## Scope confirmation

External VALIDATION was not used to select the weighting strategy and was evaluated once after
selection. HOLDOUT was never assigned, transformed, predicted, summarized, or evaluated. No new
features, tuning, calibration, threshold optimization, target encoding, ensembling, SHAP, or rules
are included.
"""
report_path = PROJECT_ROOT / "reports" / "time_decay_weighting_summary.md"
report_path.write_text(report, encoding="utf-8")
display(cv_results)
print(f"Winner: {winning_strategy} ({half_life_text})")
print(f"Validation PR-AUC: {validation_metrics.pr_auc:.6f}")
print(f"Absolute change: {absolute_change:+.6f}")
print(f"Relative change: {relative_change:+.2%}")
print(f"Final MLflow run: {final_run_id}")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print("External validation used for selection: False")
print("Final holdout status: not assigned or accessed")

,Strategy,Half-Life Days,Fold PR-AUC,Mean CV PR-AUC,Std CV PR-AUC,Mean CV ROC-AUC,Mean Fit Seconds,MLflow Run ID
0,unweighted,NaN,"(0.5542290843797867, 0.5748513290293689, 0.625...",0.584875,0.029965,0.906202,29.809155,789682d8e85c4fd295887d713c3a1556
1,half_life_90_days,90.0,"(0.5753792091437989, 0.5569121986044531, 0.615...",0.582565,0.024414,0.903198,31.051278,087bcba5344a45fcb4295931a68a51ea
2,half_life_60_days,60.0,"(0.5547203174079476, 0.5686846167265776, 0.622...",0.582122,0.029435,0.903318,29.897532,1618fcb388f64b3bb1e4e4cc1c19c6b3
3,half_life_30_days,30.0,"(0.5563361595570846, 0.5567319917943031, 0.612...",0.575246,0.026464,0.900510,28.347494,ca01d155d6a5495ca0d93c3b58f09aba


Winner: unweighted (none)
Validation PR-AUC: 0.595387
Absolute change: +0.000000
Relative change: +0.00%
Final MLflow run: fd07db6771a14f6088c9d0b9177dfd9b
Report: reports\time_decay_weighting_summary.md
External validation used for selection: False
Final holdout status: not assigned or accessed
